# Cement NPV summary

Run the cement Monte Carlo NPV distribution, deterministic NPV, and Monte Carlo ranking summaries. The distribution table includes the number and share of simulations with non-negative NPV (NPV >= 0) and the number with negative NPV. This notebook displays figures inline only; it does not save figures or CSV files.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_npv_deterministic import calculate_deterministic_cement_results
from cement.cement_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_RETROFIT_BAU_MODE,
    DEFAULT_SAMPLE_SIZE,
    simulate_cement_results,
)
from cement.cement_npv_summary_figures import (
    CEMENT_NPV_SCALE_OPTIONS,
    CEMENT_TECHNOLOGY_LABELS,
    calculate_cement_npv_rankings_from_results,
)
from npv_summary import summarize_metric_signs
from npv_summary_plots import plot_average_rank_bars, plot_mean_npv_technology_bars


## Settings

In [ ]:
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE
TECHNOLOGIES = tuple(CEMENT_TECHNOLOGY_LABELS)
NPV_SCALE = "MEUR"  # "MEUR" or "EUR/t"

if NPV_SCALE not in CEMENT_NPV_SCALE_OPTIONS:
    raise ValueError(f"Unknown NPV_SCALE: {NPV_SCALE!r}")

NPV_CONFIG = CEMENT_NPV_SCALE_OPTIONS[NPV_SCALE]
NPV_METRIC_COLUMN = str(NPV_CONFIG["metric_column"])
NPV_SCALE_FACTOR = float(NPV_CONFIG["scale"])
NPV_SUMMARY_COLUMN = str(NPV_CONFIG["summary_column"])
NPV_AXIS_LABEL = str(NPV_CONFIG["axis_label"])
NPV_RANKING_LABEL = str(NPV_CONFIG["ranking_label"])

pd.options.display.float_format = "{:,.3f}".format


## Monte Carlo NPV distribution

In [ ]:
monte_carlo_results = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=TECHNOLOGIES,
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)

mc_summary_rows = []
for technology, results in monte_carlo_results.items():
    values = np.asarray(results[NPV_METRIC_COLUMN], dtype=float) / NPV_SCALE_FACTOR
    mc_summary_rows.append(
        {
            "label": CEMENT_TECHNOLOGY_LABELS.get(technology, technology),
            "technology_type": str(np.asarray(results["technology_type"])[0]),
            NPV_SUMMARY_COLUMN: values.mean(),
            "median": np.median(values),
            "p05": np.percentile(values, 5),
            "p95": np.percentile(values, 95),
            **summarize_metric_signs(values),
        }
    )
mc_summary = pd.DataFrame(mc_summary_rows).sort_values(NPV_SUMMARY_COLUMN, ascending=False)
mc_summary_by_label = mc_summary.set_index("label")

display(mc_summary)
plot_mean_npv_technology_bars(
    values_million_eur=mc_summary_by_label[NPV_SUMMARY_COLUMN].to_dict(),
    output_path=None,
    title=f"Monte Carlo mean {NPV_RANKING_LABEL} by cement technology",
    median_values_million_eur=mc_summary_by_label["median"].to_dict(),
    lower_values_million_eur=mc_summary_by_label["p05"].to_dict(),
    upper_values_million_eur=mc_summary_by_label["p95"].to_dict(),
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    x_axis_label=NPV_AXIS_LABEL,
)
plt.show()


## Deterministic NPV

In [ ]:
deterministic_results = calculate_deterministic_cement_results(technologies=TECHNOLOGIES)
det_summary_rows = []
for technology, results in deterministic_results.items():
    det_summary_rows.append(
        {
            "label": CEMENT_TECHNOLOGY_LABELS.get(technology, technology),
            "technology_type": str(np.asarray(results["technology_type"])[0]),
            NPV_SUMMARY_COLUMN: float(np.asarray(results[NPV_METRIC_COLUMN]).item()) / NPV_SCALE_FACTOR,
        }
    )
det_summary = pd.DataFrame(det_summary_rows).sort_values(NPV_SUMMARY_COLUMN, ascending=False)

display(det_summary)
plot_mean_npv_technology_bars(
    values_million_eur=det_summary.set_index("label")[NPV_SUMMARY_COLUMN].to_dict(),
    output_path=None,
    title=f"Deterministic {NPV_RANKING_LABEL} by cement technology",
    x_axis_label=NPV_AXIS_LABEL,
)
plt.show()


## Monte Carlo NPV ranking

In [ ]:
ranking_raw, ranking_summary = calculate_cement_npv_rankings_from_results(
    results=monte_carlo_results,
    sector_name="Cement",
    npv_scale=NPV_SCALE,
)

ranking_summary_for_plot = ranking_summary.assign(
    display_label=ranking_summary["technology"].map(CEMENT_TECHNOLOGY_LABELS).fillna(ranking_summary["technology"])
)
rank_table = (
    ranking_summary_for_plot.rename(columns={"display_label": "Technology"})
    .loc[:, ["Technology", "average_rank", "probability_rank_1", "probability_top_3", "n_simulations"]]
    .rename(
        columns={
            "average_rank": "Average rank",
            "probability_rank_1": "Probability rank 1",
            "probability_top_3": "Probability top 3",
            "n_simulations": "Simulations",
        }
    )
)
display(rank_table)
plot_average_rank_bars(
    ranking_summary=ranking_summary_for_plot,
    output_path=None,
    title=f"Monte Carlo {NPV_RANKING_LABEL} Ranking",
    random_seed=RANDOM_SEED,
)
plt.show()
